# Practica 2 (10-15 min): GraphRAG-light
## Master Oficial: Big Data Science

### Objetivo
Construir un mini flujo GraphRAG sobre embeddings de una GNN entrenada rapido:
1. Seed retrieval (top-k)
2. Expansion 1-hop
3. Metricas y visualizacion

### Entregables
- Tabla con `hits@k`, `purity_expanded`, `expansion_factor` y `latency_ms`.
- Una figura del subgrafo expandido.
- Interpretacion corta (2 lineas).

### TODOs de esta practica
1. Definir `query_vec` como centroide de clase.
2. Recuperar top-k por similitud coseno.
3. Expandir 1-hop desde seeds.

### Checkpoints
- `len(seeds) == K_TOP`
- `len(expanded_nodes) >= len(seeds)`
- metricas en rango [0, 1] donde aplica

In [1]:
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

import torch
import torch.nn.functional as F
from IPython.display import display
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATv2Conv

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

K_TOP = 8
MAX_EXPANDED_NODES = 250
MAX_PLOT_NODES = 60

print(f"Device: {device} | Data dir: {DATA_DIR.resolve()}")

/home/darian/projects/GNN_MaterialesCurso/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu | Data dir: /home/darian/projects/GNN_MaterialesCurso/data


In [2]:
def load_dataset_with_fallback(data_dir: Path):
    last_error = None
    for name in ["Cora", "CiteSeer"]:
        try:
            ds = Planetoid(root=str(data_dir / "planetoid"), name=name)
            return ds, ds[0], name
        except Exception as exc:
            last_error = exc
            print(f"Aviso: no se pudo cargar {name}: {type(exc).__name__}: {exc}")
    raise RuntimeError(f"No se pudo cargar Cora ni CiteSeer: {last_error}")

dataset, data, dataset_name = load_dataset_with_fallback(DATA_DIR)
data = data.to(device)

print(f"Dataset activo: {dataset_name}")
print(f"nodes={data.num_nodes} | edges={data.num_edges} | classes={dataset.num_classes}")

Dataset activo: Cora
nodes=2708 | edges=10556 | classes=7


In [3]:
class QuickGAT(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=4, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads)
        self.gat2 = GATv2Conv(hidden_dim * heads, out_dim, heads=1)

    def forward(self, x, edge_index):
        h = F.dropout(x, p=self.dropout, training=self.training)
        h = self.gat1(h, edge_index)
        h = F.elu(h)
        emb = h
        h = F.dropout(h, p=self.dropout, training=self.training)
        logits = self.gat2(h, edge_index)
        return logits, emb


def quick_train_gat(data, in_dim, out_dim, device, epochs=60):
    model = QuickGAT(in_dim, hidden_dim=16, out_dim=out_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        logits, _ = model(data.x, data.edge_index)
        loss = criterion(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        logits, embeddings = model(data.x, data.edge_index)

    pred = logits.argmax(dim=1)
    test_acc = float((pred[data.test_mask] == data.y[data.test_mask]).float().mean().item())
    return model, embeddings.detach().cpu(), test_acc


model, embeddings, test_acc = quick_train_gat(data, dataset.num_features, dataset.num_classes, device, epochs=60)
y_cpu = data.y.detach().cpu()
train_idx = torch.where(data.train_mask.detach().cpu())[0]
test_idx = torch.where(data.test_mask.detach().cpu())[0]

print(f"Quick GAT test_acc={test_acc:.4f}")

Quick GAT test_acc=0.7880


In [ ]:
# TODO 1: query por centroide de clase
train_labels = y_cpu[train_idx]
classes, counts = torch.unique(train_labels, return_counts=True)

# 1) Elige target_class como la clase con mas nodos en train
# target_class = int(classes[counts.argmax()].item())

# 2) Define query_vec como promedio de embeddings de esa clase en train
# query_vec = embeddings[train_idx[train_labels == target_class]].mean(dim=0)

raise NotImplementedError("Completa TODO 1")

In [ ]:
# TODO 2: retrieval top-k por coseno con torch (sin sklearn)
# emb_test = embeddings[test_idx]  # (T, d)
# q = query_vec  # (d,)

# emb_test = F.normalize(emb_test, p=2, dim=1)
# q = F.normalize(q, p=2, dim=0)

# sims = emb_test @ q  # (T,)
# topk_pos = torch.topk(sims, k=K_TOP).indices
# seeds = test_idx[topk_pos].tolist()
# seed_scores = sims[topk_pos].detach().cpu().tolist()

raise NotImplementedError("Completa TODO 2")

In [ ]:
# TODO 3: expansion 1-hop
# 1) Construye mapa de vecinos desde edge_index
# 2) expanded_nodes = seeds U vecinos(seeds)
# 3) Capar expansion para no crecer demasiado

# edge_index_cpu = data.edge_index.detach().cpu()
# neighbors = defaultdict(set)
# for u, v in edge_index_cpu.t().tolist():
#     neighbors[u].add(v)
#     neighbors[v].add(u)

# expanded_nodes = list(seeds)
# for s in seeds:
#     expanded_nodes.extend(list(neighbors.get(s, set())))

# expanded_nodes = list(dict.fromkeys(expanded_nodes))  # unique preservando orden
# expanded_nodes = expanded_nodes[:MAX_EXPANDED_NODES]

# Referencia para evitar warning de import antes de completar TODO
_ = defaultdict

raise NotImplementedError("Completa TODO 3")

In [ ]:
# Checkpoints + metricas (se activan cuando completes TODOs)
seeds_var = globals().get("seeds")
expanded_var = globals().get("expanded_nodes")
target_var = globals().get("target_class")

if seeds_var is None or expanded_var is None or target_var is None:
    print("Completa TODOs 1-3 para calcular metricas")
else:
    assert len(seeds_var) == K_TOP, "Top-k incompleto"
    assert len(expanded_var) >= len(seeds_var), "Expansion invalida"

    hits_at_k = float((y_cpu[seeds_var] == target_var).float().mean().item())
    purity_expanded = float((y_cpu[expanded_var] == target_var).float().mean().item())
    expansion_factor = len(expanded_var) / len(seeds_var)

    metrics_df = pd.DataFrame([
        {
            "dataset": dataset_name,
            "target_class": int(target_var),
            "hits@k": round(hits_at_k, 4),
            "purity_expanded": round(purity_expanded, 4),
            "expansion_factor": round(expansion_factor, 2),
        }
    ])
    display(metrics_df)
    print("Checkpoint metricas OK")

In [ ]:
# Visualizacion subgrafo (cuando TODOs completos)
seeds_var = globals().get("seeds")
seed_scores_var = globals().get("seed_scores")
expanded_var = globals().get("expanded_nodes")
neighbors_var = globals().get("neighbors")
target_var = globals().get("target_class")

if (
    seeds_var is None
    or seed_scores_var is None
    or expanded_var is None
    or neighbors_var is None
    or target_var is None
):
    print("Completa TODOs 1-3 para visualizar")
else:
    node_set = list(seeds_var)
    for n in expanded_var:
        if n not in node_set:
            node_set.append(int(n))
        if len(node_set) >= MAX_PLOT_NODES:
            break

    G = nx.Graph()
    G.add_nodes_from(node_set)
    for u in node_set:
        for v in neighbors_var.get(u, set()):
            if v in G:
                G.add_edge(u, v)

    score_map = {int(n): 0.0 for n in node_set}
    for n, s in zip(seeds_var, seed_scores_var):
        score_map[int(n)] = float(s)

    node_colors = [int(y_cpu[n]) for n in G.nodes()]
    node_sizes = [220 + 1200 * max(0.0, score_map[int(n)]) for n in G.nodes()]

    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.8)
    nodes = nx.draw_networkx_nodes(
        G,
        pos,
        node_color=node_colors,
        node_size=node_sizes,
        cmap=plt.cm.tab10,
        alpha=0.9,
        edgecolors="black",
        linewidths=0.3,
    )
    plt.colorbar(nodes, label="class label")
    plt.title(f"GraphRAG-light | dataset={dataset_name} | target_class={int(target_var)}")
    plt.axis("off")
    plt.show()

## Interpretacion breve

Incluye en tu entrega:
1. Tabla de metricas (`hits@k`, `purity_expanded`, `expansion_factor`).
2. Captura o descripcion breve del subgrafo recuperado/expandido.

Responde en 2-3 lineas:
1. Como cambia el contexto al expandir por vecinos?
2. La pureza sube o baja al expandir? Por que podria pasar eso?